# Dripito Rev-B — A Priori MC Error Budget

Predicts MAPE and Bland-Altman LoA for the dual-beam drop-volume
measurement **before** bench data is collected. This notebook is
the source of truth for the predicted figure cited in
`docs/architecture.md` and `docs/limitations.md`.

**The discipline that makes this useful.** Run this notebook,
commit the resulting `fig_error_budget.png`, **then** take bench
data. The git timeline is what turns the prediction into a
defensible claim rather than post-hoc fitting.

**Inputs.** None — fully synthetic from the parameters in
`scripts.error_budget.default_params()`. Every parameter traces
to a measurement, a datasheet, or a flagged placeholder.

**Outputs.**

- `figures/fig_error_budget.png` — 3-panel: ablation · predicted Bland-Altman · calibration convergence
- `figures/error_budget_summary.csv` — per-rate predicted bias / SD / LoA / MAPE
- `figures/error_budget_ablation.csv` — source-by-source ablation table
- `figures/error_budget_convergence.csv` — SD(V_cal) vs N_drops
- `data/sample/error_budget_synthetic.csv` — stratified sample for cold-clone reproducibility

Regenerate via:

```bash
cd analysis
docker compose up regenerate-error-budget
```

## 0. Concepts referenced in this notebook

Glossary of the acronyms and statistical objects this analysis uses.

### Statistical terms

- **MC (Monte Carlo simulation).** Draws random inputs from realistic distributions, runs the system forward many times, looks at the distribution of outputs. Used when the math is too tangled for closed-form error propagation.
- **CV (Coefficient of Variation).** Standard deviation divided by mean, in %. *"How spread out is this random variable, relative to its size?"* A drop-volume CV of 8% means 1-sigma scatter is 8% of the mean drop volume.
- **MAPE (Mean Absolute Percentage Error).** For each drop: `|estimate − truth| / truth × 100%`. Average across all drops. Single scalar summary of relative accuracy.
- **Bland-Altman plot.** The standard agreement plot for two measurement methods. X = mean of the two methods; Y = their difference. Horizontal lines at bias and ±1.96·SD (the 95% Limits of Agreement).
- **LoA (Limits of Agreement).** ±1.96 × SD of `(estimate − truth)`. The band inside which 95% of paired measurements should fall.
- **Ablation.** A what-if analysis. Force one noise source's SD to zero, re-run the simulation, measure how much the total error shrinks. The shrinkage = that source's attributable contribution.

### Architecture terms

- **Dual-beam transit measurement.** Two parallel IR beams separated by `d_nominal` (printed sensor-arm geometry). A drop falls through TOP beam, then BOT beam. Firmware records four edge times (`tT_in, tT_out, tB_in, tB_out`).
- **Transit `dt`.** `tB_in − tT_in` = time for the drop's leading edge to travel from TOP beam to BOT beam.
- **Shadow `tau`.** `tT_out − tT_in` = time the drop spends blocking the TOP beam.
- **Velocity-cancellation property (the architecture's headline result).** The firmware estimates drop diameter as `D = d_nominal × (tau / dt)`. Both `tau` and `dt` scale inversely with the drop's velocity, so their ratio is **velocity-invariant**. The device does not need to know how fast the drop is moving to estimate its volume. This is why per-drop velocity CV contributes ~0 to the error budget in panel (a).
- **Print tolerance.** The Bambu Lab 3D-print tolerance on the sensor arm produces per-board variation in the actual `d`. The firmware bakes in `d_nominal = 10.0 mm`, so per-board geometric scatter creates a systematic bias that the model surfaces as the dominant error source.
- **Calibration `V_cal`.** Firmware boot sequence: collect `CAL_N` drops, sort, trim lowest + highest, average the rest. The resulting `V_cal` is the session's locked drop-volume reference. Subsequent drops are compared to it for alarm logic and flow-rate integration.

In [ ]:
# Parameters (overridable by papermill -p)
FIGURES_DIR = "../figures"
SAMPLE_DIR = "../../data/sample"
N_CAMPAIGNS = 1000
ABLATION_CAMPAIGNS = 400
CONVERGENCE_BOOTSTRAP = 2000
RANDOM_SEED = 20260513  # bench date — keeps reruns byte-identical

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ANALYSIS_ROOT = Path.cwd().resolve()
if (ANALYSIS_ROOT / "scripts").is_dir():
    sys.path.insert(0, str(ANALYSIS_ROOT))
else:
    sys.path.insert(0, str(ANALYSIS_ROOT.parent))

from scripts.error_budget import (
    ablation,
    calibration_convergence,
    default_params,
    make_figure,
    simulate,
    summarise,
    write_sample_csv,
)

FIGURES_DIR_P = Path(FIGURES_DIR).resolve()
SAMPLE_DIR_P = Path(SAMPLE_DIR).resolve()
FIGURES_DIR_P.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR_P.mkdir(parents=True, exist_ok=True)
print(f"FIGURES_DIR = {FIGURES_DIR_P}")
print(f"SAMPLE_DIR  = {SAMPLE_DIR_P}")

## 1. Parameters — the assumption table

Every value below is the assumption table for `docs/architecture.md`.
Refine after bench:

- `d_sd_mm` — currently set to the Bambu Lab print tolerance. If you start individually-calibrating boards (storing measured `d_actual` in flash), this term shrinks to the residual calibration uncertainty.
- `adc_sample_period_us` and `threshold_jitter_us` — read from the firmware ADC config and threshold-crossing implementation.
- `drop_volume_cv` — fit from per-drop gravimetric distribution post-bench.

In [ ]:
params = default_params()
for k, v in params.__dict__.items():
    print(f"  {k:28s} = {v}")

## 2. Monte Carlo simulation

Hierarchical: per-board / per-run `d_actual` sampled **once per run** (constant across that run's drops), all other noise sampled per drop. This produces the within-run / between-run Bland-Altman structure the bench either confirms or refutes.

In [ ]:
df = simulate(params, n_campaigns=N_CAMPAIGNS, seed=RANDOM_SEED)
print(f"{len(df):,} synthetic drops across {df['run_id'].nunique():,} simulated runs")

summary = summarise(df)
summary

## 3. Ablation: which noise source dominates?

For each ablatable source, zero its SD and re-run the MC. The delta in MAPE between baseline and ablated = attributable contribution of that source.

**Why some sources contribute ~0:**

- *Drop volume CV* — the device measures whatever volume the drop actually is; drop heterogeneity affects truth and estimate identically, cancelling in the BA difference.
- *Drop velocity CV* — velocity cancels in the firmware's `(tau / dt)` ratio inversion. This is the architecture's velocity-invariance property.
- *Alignment drift (per-run)* — set to zero by design: the printed sensor arm fixes `d` rigidly across chamber remounts.

In [ ]:
abl = ablation(params, n_campaigns=ABLATION_CAMPAIGNS, seed=RANDOM_SEED)
pooled = abl[abl["rate_mlh"] == "all"].sort_values("delta_vs_baseline_pct", ascending=False)
pooled[["source", "mape_pct", "delta_vs_baseline_pct"]]

## 4. Calibration convergence — per-board scenario

The firmware's boot sequence collects `CAL_N` drops, sorts them, trims `CAL_TRIM_LO=1` from the low end and `CAL_TRIM_HI=1` from the high end, averages the rest. The resulting `V_cal` becomes the session's locked drop-volume reference.

This analysis predicts the SD of `V_cal` (across many calibration attempts on the SAME board) as a function of how many drops are used. Print tolerance is zeroed because the calibration is against this board's actual drops — the firmware doesn't care about the absolute `d`, only that subsequent drops match `V_cal`.

Practical purpose: choose `CAL_N` to hit a target accuracy in a reasonable time, before transitioning into the energy-saver / measurement phase.

In [ ]:
conv = calibration_convergence(params, n_drops_max=30, n_bootstrap=CONVERGENCE_BOOTSTRAP, seed=RANDOM_SEED)
# Show the firmware default + a few neighbouring choices
highlight = conv[conv["n_drops"].isin([5, 8, 10, 15, 20, 30])]
highlight

## 5. Figure

In [ ]:
fig = make_figure(df, abl, params, convergence_df=conv)
fig.savefig(FIGURES_DIR_P / "fig_error_budget.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Persist summary tables and sample data

In [ ]:
summary.to_csv(FIGURES_DIR_P / "error_budget_summary.csv", index=False)
abl.to_csv(FIGURES_DIR_P / "error_budget_ablation.csv", index=False)
conv.to_csv(FIGURES_DIR_P / "error_budget_convergence.csv", index=False)
write_sample_csv(df, str(SAMPLE_DIR_P / "error_budget_synthetic.csv"), seed=RANDOM_SEED)
print(f"Wrote {FIGURES_DIR_P / 'error_budget_summary.csv'}")
print(f"Wrote {FIGURES_DIR_P / 'error_budget_ablation.csv'}")
print(f"Wrote {FIGURES_DIR_P / 'error_budget_convergence.csv'}")
print(f"Wrote {SAMPLE_DIR_P / 'error_budget_synthetic.csv'}")

## 7. Bench overlay (post-bench)

Once `data/raw/` contains real per-drop measurements paired with per-drop gravimetric truth, build a DataFrame with columns `v_est_ul` and `v_true_ul` and call `overlay_bench_points(ax, df)` on panel (b) of a fresh figure. The overlay should either land inside the predicted LoA bands or reveal which assumption was wrong.

```python
from scripts.error_budget import overlay_bench_points
fig = make_figure(df, abl, params, convergence_df=conv)
ax_b = fig.axes[1]  # the Bland-Altman panel
overlay_bench_points(ax_b, bench_df)
fig.savefig(FIGURES_DIR_P / 'fig_error_budget_with_bench.png', dpi=150, bbox_inches='tight')
```

Commit that overlay figure separately. Two committed figures — predicted-only (timestamped pre-bench) and predicted-plus-bench (post-bench) — give the git history the predict→measure story.

---

## Provenance

Regenerate identically via:

```bash
cd analysis
docker compose up regenerate-error-budget
```

`RANDOM_SEED` is fixed to `20260513` (bench date) so reruns are byte-identical until a parameter changes.